# Basketball Trajectory Export

This notebook processes basketball game footage using SAM2 tracking and exports player/ball trajectories as JSON for downstream play clustering analysis.

**Pipeline:**
1. Detect players on frame 0 using RF-DETR
2. Classify teams using TeamClassifier
3. Track players across video using SAM2
4. Detect court keypoints and compute homography per frame
5. Transform pixel coordinates to court coordinates (feet)
6. Export trajectories as JSON

**Outputs:**
- `{game_id}_trajectories.json` - Player and ball trajectories in court coordinates

## Setup & Installation

In [ ]:
# Install dependencies
%pip install -q inference supervision sports roboflow
%pip install -q git+https://github.com/facebookresearch/sam2.git

In [ ]:
# Mount Google Drive for video storage and output
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Download SAM2 checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt -O sam2_hiera_large.pt
!wget -q https://raw.githubusercontent.com/facebookresearch/segment-anything-2/main/sam2_configs/sam2_hiera_l.yaml -O sam2_hiera_l.yaml

## Configuration

In [ ]:
from pathlib import Path
import os

# === CONFIGURE THESE ===

# Roboflow API key (get from https://app.roboflow.com/settings/api)
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # @param {type:"string"}

# Input video path (on Google Drive)
SOURCE_VIDEO_PATH = "/content/drive/MyDrive/hoopstats/videos/game1.mp4"  # @param {type:"string"}

# Output directory for trajectories
OUTPUT_DIR = "/content/drive/MyDrive/hoopstats/raw_trajectories"  # @param {type:"string"}

# Game identifier (used in output filename)
GAME_ID = "game1"  # @param {type:"string"}

# SAM2 checkpoint paths
SAM2_CHECKPOINT = "sam2_hiera_large.pt"
SAM2_CONFIG = "sam2_hiera_l.yaml"

# === END CONFIG ===

# Ensure output directory exists
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Validate video exists
if not Path(SOURCE_VIDEO_PATH).exists():
    raise FileNotFoundError(f"Video not found: {SOURCE_VIDEO_PATH}")

print(f"Video: {SOURCE_VIDEO_PATH}")
print(f"Output: {OUTPUT_DIR}/{GAME_ID}_trajectories.json")

## Load Models

In [ ]:
from inference import get_model

# Model IDs from Roboflow
PLAYER_DETECTION_MODEL_ID = "rf-detr-basketball-3wdkj/3"
KEYPOINT_DETECTION_MODEL_ID = "basketball-court-detection-2/14"

# Detection thresholds
PLAYER_DETECTION_MODEL_CONFIDENCE = 0.3
PLAYER_DETECTION_MODEL_IOU_THRESHOLD = 0.5
KEYPOINT_DETECTION_MODEL_CONFIDENCE = 0.3
KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE = 0.5

# Class IDs
PLAYER_CLASS_IDS = [3, 4, 5, 6, 7]  # player, player-in-possession, jump-shot, layup-dunk, shot-block
BALL_CLASS_ID = 0

print("Loading RF-DETR player detection model...")
PLAYER_DETECTION_MODEL = get_model(
    model_id=PLAYER_DETECTION_MODEL_ID,
    api_key=ROBOFLOW_API_KEY
)

print("Loading keypoint detection model...")
KEYPOINT_DETECTION_MODEL = get_model(
    model_id=KEYPOINT_DETECTION_MODEL_ID,
    api_key=ROBOFLOW_API_KEY
)

print("Models loaded successfully!")

In [ ]:
from sam2.build_sam import build_sam2_camera_predictor

print("Loading SAM2 predictor...")
sam2_predictor = build_sam2_camera_predictor(SAM2_CONFIG, SAM2_CHECKPOINT)
print("SAM2 loaded successfully!")

## SAM2 Tracker Class

In [ ]:
from __future__ import annotations
import numpy as np
import torch
import supervision as sv


class SAM2Tracker:
    """SAM2-based object tracker for video."""

    def __init__(self, predictor) -> None:
        self.predictor = predictor
        self._prompted = False

    def prompt_first_frame(self, frame: np.ndarray, detections: sv.Detections) -> None:
        """Initialize tracking with detections from the first frame."""
        if len(detections) == 0:
            raise ValueError("detections must contain at least one box")

        # Assign tracker IDs if not present
        if detections.tracker_id is None:
            detections.tracker_id = np.arange(1, len(detections) + 1)

        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            self.predictor.load_first_frame(frame)
            for xyxy, obj_id in zip(detections.xyxy, detections.tracker_id):
                bbox = np.asarray([xyxy], dtype=np.float32)
                self.predictor.add_new_prompt(
                    frame_idx=0,
                    obj_id=int(obj_id),
                    bbox=bbox,
                )

        self._prompted = True
        print(f"SAM2 prompted with {len(detections)} objects")

    def propagate(self, frame: np.ndarray) -> sv.Detections:
        """Propagate tracking to a new frame."""
        if not self._prompted:
            raise RuntimeError("Call prompt_first_frame before propagate")

        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            tracker_ids, mask_logits = self.predictor.track(frame)

        tracker_ids = np.asarray(tracker_ids, dtype=np.int32)
        masks = (mask_logits > 0.0).cpu().numpy()
        masks = np.squeeze(masks).astype(bool)

        # Ensure masks has batch dimension
        if masks.ndim == 2:
            masks = masks[None, ...]

        # Filter noisy mask segments
        masks = np.array([
            sv.filter_segments_by_distance(mask, relative_distance=0.03, mode="edge")
            for mask in masks
        ])

        # Convert masks to bounding boxes
        xyxy = sv.mask_to_xyxy(masks=masks)
        detections = sv.Detections(xyxy=xyxy, mask=masks, tracker_id=tracker_ids)

        return detections

## Initialize Video and First Frame Detection

In [ ]:
from sports.common.team import TeamClassifier
from sports.common.view import ViewTransformer
from sports.basketball import CourtConfiguration, League
from sports.common.core import MeasurementUnit

# Get video info
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
print(f"Video: {video_info.width}x{video_info.height} @ {video_info.fps}fps, {video_info.total_frames} frames")

# Court configuration
config = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)

# Get first frame
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
first_frame = next(frame_generator)

# Detect players on first frame
result = PLAYER_DETECTION_MODEL.infer(
    first_frame,
    confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
    iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD,
    class_agnostic_nms=True
)[0]
detections = sv.Detections.from_inference(result)

# Filter to player classes only
player_mask = np.isin(detections.class_id, PLAYER_CLASS_IDS)
player_detections = detections[player_mask]
player_detections.tracker_id = np.arange(1, len(player_detections) + 1)

print(f"Detected {len(player_detections)} players on first frame")

## Team Classification

In [ ]:
# Train team classifier on crops from first frame
# Scale boxes to get jersey crops (0.4 factor focuses on jersey area)
boxes = sv.scale_boxes(xyxy=player_detections.xyxy, factor=0.4)
crops = [sv.crop_image(first_frame, box) for box in boxes]

print(f"Training team classifier on {len(crops)} player crops...")
team_classifier = TeamClassifier(device="cuda")
team_classifier.fit(crops)

# Predict teams for first frame detections
TEAMS = np.array(team_classifier.predict(crops))
print(f"Team 0: {np.sum(TEAMS == 0)} players, Team 1: {np.sum(TEAMS == 1)} players")

# Create tracker_id -> team mapping
tracker_team_map = {int(tid): int(team) for tid, team in zip(player_detections.tracker_id, TEAMS)}
print(f"Tracker-Team mapping: {tracker_team_map}")

## Initialize SAM2 Tracker

In [ ]:
# Initialize SAM2 tracker with first frame detections
tracker = SAM2Tracker(sam2_predictor)
tracker.prompt_first_frame(first_frame, player_detections)

## Process Video - Extract Trajectories

In [ ]:
from tqdm import tqdm
import json

# Storage for trajectories
# player trajectories: tracker_id -> list of [x, y, frame_idx]
player_trajectories = {int(tid): [] for tid in player_detections.tracker_id}

# ball trajectory: list of [x, y, frame_idx]
ball_trajectory = []

# Track frames where homography failed
failed_frames = []

# Process first frame (already detected)
kp_result = KEYPOINT_DETECTION_MODEL.infer(first_frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE)[0]
key_points = sv.KeyPoints.from_inference(kp_result)

if key_points.confidence is not None and len(key_points.confidence) > 0:
    landmarks_mask = key_points.confidence[0] > KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE
    
    if np.count_nonzero(landmarks_mask) >= 4:
        court_landmarks = np.array(config.vertices)[landmarks_mask]
        frame_landmarks = key_points[:, landmarks_mask].xy[0]
        
        transformer = ViewTransformer(source=frame_landmarks, target=court_landmarks)
        
        # Transform player positions
        frame_xy = player_detections.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
        court_xy = transformer.transform_points(points=frame_xy)
        
        for tid, xy in zip(player_detections.tracker_id, court_xy):
            player_trajectories[int(tid)].append([float(xy[0]), float(xy[1]), 0])
        
        # Ball detection for first frame
        ball_mask = detections.class_id == BALL_CLASS_ID
        ball_dets = detections[ball_mask]
        if len(ball_dets) > 0:
            ball_xy = ball_dets.get_anchors_coordinates(anchor=sv.Position.CENTER)
            ball_court = transformer.transform_points(points=ball_xy)
            ball_trajectory.append([float(ball_court[0][0]), float(ball_court[0][1]), 0])

print(f"Starting trajectory extraction for {video_info.total_frames} frames...")

In [ ]:
# Process remaining frames
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
next(frame_generator)  # Skip first frame (already processed)

for frame_idx, frame in tqdm(enumerate(frame_generator, start=1), total=video_info.total_frames - 1):
    # SAM2 propagate player tracking
    tracked_detections = tracker.propagate(frame)
    
    # Keypoint detection for homography
    kp_result = KEYPOINT_DETECTION_MODEL.infer(frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE)[0]
    key_points = sv.KeyPoints.from_inference(kp_result)
    
    # Check for valid keypoints
    if key_points.confidence is None or len(key_points.confidence) == 0:
        failed_frames.append(frame_idx)
        continue
    
    landmarks_mask = key_points.confidence[0] > KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE
    
    if np.count_nonzero(landmarks_mask) < 4:
        failed_frames.append(frame_idx)
        continue
    
    # Compute homography
    court_landmarks = np.array(config.vertices)[landmarks_mask]
    frame_landmarks = key_points[:, landmarks_mask].xy[0]
    
    try:
        transformer = ViewTransformer(source=frame_landmarks, target=court_landmarks)
    except Exception as e:
        failed_frames.append(frame_idx)
        continue
    
    # Transform player positions to court coordinates
    if len(tracked_detections) > 0:
        frame_xy = tracked_detections.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
        court_xy = transformer.transform_points(points=frame_xy)
        
        for tid, xy in zip(tracked_detections.tracker_id, court_xy):
            tid = int(tid)
            if tid in player_trajectories:
                player_trajectories[tid].append([float(xy[0]), float(xy[1]), frame_idx])
    
    # Ball detection (run RF-DETR separately since SAM2 doesn't track ball well)
    ball_result = PLAYER_DETECTION_MODEL.infer(
        frame,
        confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
        iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD
    )[0]
    ball_dets = sv.Detections.from_inference(ball_result)
    ball_dets = ball_dets[ball_dets.class_id == BALL_CLASS_ID]
    
    if len(ball_dets) > 0:
        ball_xy = ball_dets.get_anchors_coordinates(anchor=sv.Position.CENTER)
        ball_court = transformer.transform_points(points=ball_xy)
        # Take first ball detection if multiple
        ball_trajectory.append([float(ball_court[0][0]), float(ball_court[0][1]), frame_idx])

print(f"\nProcessing complete!")
print(f"  Failed homography on {len(failed_frames)} frames ({100*len(failed_frames)/video_info.total_frames:.1f}%)")
print(f"  Ball detections: {len(ball_trajectory)} frames")
for tid, traj in player_trajectories.items():
    print(f"  Player {tid} (Team {tracker_team_map.get(tid, '?')}): {len(traj)} positions")

## Export Trajectories to JSON

In [ ]:
# Build export data structure
export_data = {
    "game_id": GAME_ID,
    "source_video": str(Path(SOURCE_VIDEO_PATH).name),
    "fps": video_info.fps,
    "total_frames": video_info.total_frames,
    "width": video_info.width,
    "height": video_info.height,
    "court_config": {
        "league": "NBA",
        "unit": "feet",
        "length": 94,
        "width": 50
    },
    "players": {
        str(tid): {
            "team": tracker_team_map.get(tid, -1),
            "trajectory": traj
        }
        for tid, traj in player_trajectories.items()
    },
    "ball": {
        "trajectory": ball_trajectory
    },
    "metadata": {
        "failed_homography_frames": len(failed_frames),
        "failed_homography_percentage": round(100 * len(failed_frames) / video_info.total_frames, 2)
    }
}

# Save to file
output_path = Path(OUTPUT_DIR) / f"{GAME_ID}_trajectories.json"
with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"Trajectories exported to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

## Visualization (Optional)

In [ ]:
from sports.basketball import draw_court, draw_points_on_court, draw_paths_on_court

# Draw all player trajectories on court
court = draw_court(config=config)

# Team colors
TEAM_COLORS = {
    0: sv.Color.from_hex("#00FF00"),  # Green for team 0
    1: sv.Color.from_hex("#FF0000"),  # Red for team 1
}

for tid, data in export_data["players"].items():
    traj = np.array(data["trajectory"])
    if len(traj) > 0:
        team = data["team"]
        color = TEAM_COLORS.get(team, sv.Color.GRAY)
        court = draw_paths_on_court(
            config=config,
            paths=[traj[:, :2]],  # x, y only
            color=color,
            court=court
        )

# Draw ball trajectory
ball_traj = np.array(export_data["ball"]["trajectory"])
if len(ball_traj) > 0:
    court = draw_paths_on_court(
        config=config,
        paths=[ball_traj[:, :2]],
        color=sv.Color.from_hex("#FFA500"),  # Orange for ball
        court=court
    )

sv.plot_image(court)

## Batch Processing (Multiple Videos)

In [ ]:
# Uncomment and modify to process multiple videos
"""
import os

VIDEO_DIR = "/content/drive/MyDrive/hoopstats/videos"
OUTPUT_DIR = "/content/drive/MyDrive/hoopstats/raw_trajectories"

video_files = list(Path(VIDEO_DIR).glob("*.mp4"))
print(f"Found {len(video_files)} videos to process")

for video_path in video_files:
    game_id = video_path.stem
    output_path = Path(OUTPUT_DIR) / f"{game_id}_trajectories.json"
    
    if output_path.exists():
        print(f"Skipping {game_id} - already processed")
        continue
    
    print(f"\nProcessing: {game_id}")
    # ... (copy the processing code from above)
"""